In [ ]:
from spark_esri import spark_start, spark_stop

In [ ]:
import os
import arcpy
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [ ]:
extra_java_options = "-XX:+UseCompressedOops -XX:+AggressiveHeap"

config = {
    "spark.driver.memory": "31G",
    "spark.executor.memory": "31G",
    "spark.driver.extraJavaOptions": extra_java_options,
    "spark.executor.extraJavaOptions": extra_java_options,
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryo.unsafe": True,
    "spark.memory.offHeap.enable": True,
    "spark.memory.offHeap.size": "31G",
}

spark_stop()
spark = spark_start(config=config)

In [ ]:
@F.udf(returnType=T.StringType())
def to_wkt(x:float,y:float) -> str:
    pt = arcpy.Point(x,y)
    return arcpy.PointGeometry(pt).WKT

In [ ]:
spark\
    .range(10_000)\
    .withColumn("x",F.rand())\
    .withColumn("y",F.rand())\
    .withColumn("wkt", to_wkt("x","y"))\
    .write\
    .parquet(os.path.join("C:",os.sep,"temp","temp.parquet"), mode="overwrite")

In [ ]:
@F.udf(returnType=T.IntegerType())
def col_len(col:list) -> int:
    return len(col)

In [ ]:
spark\
    .range(10_000)\
    .withColumn("tid",(F.rand()*10).cast("int"))\
    .withColumn("xy",F.struct(F.rand().alias("x"),F.rand().alias("y")))\
    .groupBy("tid")\
    .agg(F.collect_list("xy").alias("col"))\
    .select("tid",col_len("col").alias("len"))\
    .show(truncate=False)